# verl 指标收集体系总览

verl 采用 三层指标收集架构：

In [ ]:
┌─────────────────────────────────────────────────────────────────────┐
│  第一层: 实验跟踪 (Experiment Tracking)                              │
│  wandb / mlflow / swanlab / tensorboard / console                    │
│  → 记录算法指标 (reward, loss, KL, accuracy)                        │
│  → 记录系统指标 (throughput, step_time, GPU利用率)                   │
├─────────────────────────────────────────────────────────────────────┤
│  第二层: 系统监控 (System Monitoring)                                │
│  Prometheus + Grafana + Ray Dashboard                                │
│  → vLLM/SGLang 推理引擎实时指标                                       │
│  → 长尾问题检测、资源空闲分析                                          │
├─────────────────────────────────────────────────────────────────────┤
│  第三层: 性能分析 (Performance Profiling)                            │
│  torch.profiler / Megatron Profiler / DistProfiler                   │
│  → 计算与通信时间分解                                                 │
│  → MFU (Model FLOPs Utilization) 计算                                │
└─────────────────────────────────────────────────────────────────────┘

# 第一层：实验跟踪系统
## 核心实现：verl/utils/tracking.py

In [ ]:
class Tracking:
    """
    统一跟踪接口，支持多种后端
    """
    supported_backend = [
        "wandb",        # Weights & Biases (最常用)
        "mlflow",       # MLflow (本地化)
        "swanlab",      # 国产实验平台
        "vemlp_wandb",  # 火山引擎 WandB
        "tensorboard",  # TensorBoard
        "console",      # 控制台输出
        "clearml",      # ClearML
        "trackio",      # TrackIO
        "file",         # JSONL 文件
    ]

### 使用方式

In [ ]:
# 配置示例 (命令行)
python -m verl.trainer.main_ppo \
    trainer.logger='["console","wandb"]' \
    trainer.project_name="verl_gsm8k" \
    trainer.experiment_name="qwen2.7b_grpo" \
    ...

### 记录的典型指标

| 指标类别     | 具体指标                        | 含义         |
| -------- | --------------------------- | ---------- |
| **算法指标** | `reward/mean`, `reward/std` | 奖励均值/标准差   |
|          | `loss/ppo`, `loss/entropy`  | PPO 损失/熵损失 |
|          | `kl/mean`, `kl/max`         | KL 散度      |
|          | `response_length/mean`      | 生成长度       |
| **系统指标** | `throughput/sample_per_sec` | 每秒样本数      |
|          | `step_time`                 | 每步耗时       |
|          | `gpu/memory_allocated`      | GPU 显存占用   |
| **验证指标** | `val/reward`                | 验证集奖励      |
|          | `val/accuracy`              | 验证集准确率     |


### 代码中的调用



In [ ]:
# 在 Trainer 中初始化
self.logger = Tracking(
    project_name=config.project_name,
    experiment_name=config.experiment_name,
    default_backend=config.logger,  # ["console", "wandb"]
    config=config
)

# 每步记录指标
self.logger.log({
    "reward/mean": reward_mean,
    "loss/ppo": ppo_loss,
    "kl/mean": kl_mean,
    "throughput/sample_per_sec": samples_per_sec,
}, step=global_step)

# 第二层：系统监控（Prometheus + Grafana）
## 架构设计

In [ ]:
┌─────────────────────────────────────────────────────────────┐
│                    verl Async Rollout Mode                    │
│                                                             │
│  ┌─────────────┐    ┌─────────────┐    ┌─────────────┐     │
│  │ vLLM Server │    │ vLLM Server │    │ vLLM Server │     │
│  │  (GPU 0-3)  │    │  (GPU 4-7)  │    │  (GPU 8-11) │     │
│  │ :8000/metrics│   │ :8001/metrics│   │ :8002/metrics│    │
│  └──────┬──────┘    └──────┬──────┘    └──────┬──────┘     │
│         │                  │                  │             │
│         └──────────────────┼──────────────────┘             │
│                            ↓                                │
│                   ┌─────────────────┐                        │
│                   │   Prometheus    │                        │
│                   │  (自动配置发现)  │                        │
│                   │  抓取各端口指标  │                        │
│                   └────────┬────────┘                        │
│                            ↓                                │
│                   ┌─────────────────┐                        │
│                   │     Grafana     │                        │
│                   │   (可视化面板)   │                        │
│                   └─────────────────┘                        │
└─────────────────────────────────────────────────────────────┘

## 监控的关键指标

| 面板                                 | 指标           | 用途         |
| ---------------------------------- | ------------ | ---------- |
| **vLLM Scheduler State**           | 待处理请求数       | 检测生成瓶颈     |
| **vLLM Request Generation Length** | 各请求生成长度分布    | **检测长尾问题** |
| **Cache Hit Rate**                 | KV Cache 命中率 | 评估内存效率     |
| **Throughput**                     | tokens/sec   | 生成吞吐       |

## 长尾问题检测示例
### 从 Grafana 面板可以清晰看到：
- 这是 verl 性能调优最重要的观测手段之一。

In [ ]:
正常生成:                    长尾生成:
┌─────────┐                  ┌─────────┐
│ ▓▓▓▓▓▓▓ │  大多数请求      │ ▓▓▓░░░░░ │  少数请求极长
│ ▓▓▓▓▓▓░ │  长度集中        │ ▓▓▓▓▓▓▓▓ │  拖慢整批
│ ▓▓▓▓▓░░ │                  │ ▓▓▓▓▓▓▓░ │
│ ▓▓▓▓░░░ │                  │ ▓▓▓▓▓▓░░ │
└─────────┘                  └─────────┘
  0-2000 tokens                0-20000+ tokens

# 第三层：性能分析（Profiling）
## MFU 计算

In [ ]:
# verl/workers/engine_workers.py

class TrainingWorker:
    def _postprocess_output(self, output, delta_time):
        """
        计算吞吐量和 MFU
        """
        # 1. 计算全局 token 数
        global_tokens = self._compute_global_tokens(output)
        
        # 2. 计算吞吐量
        throughput = global_tokens / delta_time  # tokens/sec
        
        # 3. 计算 MFU
        # MFU = 实际吞吐 / 理论峰值吞吐
        mfu = throughput / self.flops_counter.theoretical_peak_throughput
        
        return {
            "throughput/tokens_per_sec": throughput,
            "performance/mfu": mfu,
            "step_time": delta_time,
        }

## Megatron Profiler

In [ ]:
# 启用 Megatron Profiler
actor_rollout_ref.actor.profile.use_profile=True \
actor_rollout_ref.actor.profile.profile_ranks=[0] \
actor_rollout_ref.actor.profile.step_start=0 \
actor_rollout_ref.actor.profile.step_end=1 \
actor_rollout_ref.actor.profile.save_path="./profile"

生成 Chrome Trace 文件，可在 chrome://tracing 中查看：

In [ ]:
时间轴:
├─[0ms-50ms]   generate_sequences (vLLM 生成)
├─[50ms-80ms]  compute_ref_log_prob (Reference 前向)
├─[80ms-110ms] compute_values (Critic 前向)
├─[110ms-150ms] compute_rm_score (Reward 前向)
├─[150ms-500ms] update_actor (Actor 训练, 反向+优化)
└─[500ms-550ms] update_critic (Critic 训练)

# 指标收集的完整调用链

In [ ]:
Ray Driver (PPO Trainer)
    │
    ├── 每训练步 ───────────────────────────────┐
    │                                          │
    │  1. Rollout 阶段                         │
    │     ├── generate_sequences()             │
    │     │   └── 记录: rollout_time,          │
    │     │       response_length (mean/max)   │
    │     ├── compute_ref_log_prob()           │
    │     └── compute_values()                 │
    │                                          │
    │  2. Reward 阶段                          │
    │     └── compute_rm_score()               │
    │         └── 记录: reward/mean, reward/std │
    │                                          │
    │  3. 训练阶段                              │
    │     ├── update_actor()                   │
    │     │   └── 记录: loss/ppo, kl/mean,     │
    │     │       entropy_loss, step_time      │
    │     └── update_critic()                  │
    │         └── 记录: loss/value             │
    │                                          │
    │  4. 聚合 & 记录 ──────────────────────────┤
    │     ├── self.logger.log({...}, step)     │
    │     │   └── 发送到 wandb/mlflow/...       │
    │     ├── Prometheus 抓取 vLLM 指标         │
    │     └── Ray Dashboard 更新集群状态        │
    │                                          │
    └── 验证步 (每 N 步) ──────────────────────┤
        ├── 生成验证样本                        │
        ├── ValidationGenerationsLogger         │
        │   └── 记录 val/generations 表格        │
        └── 记录 val/reward, val/accuracy       │

# 效果达标判断标准
verl 通过以下维度判断系统效果是否达标：

| 维度       | 达标标准                      | 监控位置                 |
| -------- | ------------------------- | -------------------- |
| **算法效果** | reward 持续上升，KL 稳定         | wandb/mlflow 曲线      |
| **训练速度** | throughput 达到理论值的 80%+    | MFU 指标               |
| **系统稳定** | 无 OOM，step\_time 波动 < 10% | console + Prometheus |
| **生成质量** | 无长尾拖慢，Cache 命中率 > 90%     | Grafana 面板           |
| **扩展性**  | 线性扩展度 > 80% (多节点)         | throughput vs GPU 数  |


# 关键配置文件总结

In [ ]:
# 1. 启用 wandb 跟踪
export WANDB_API_KEY=xxx
trainer.logger='["console","wandb"]' \
trainer.project_name="my_project" \
trainer.experiment_name="grpo_qwen"

# 2. 启用 Prometheus + Grafana 监控 (Async Rollout)
actor_rollout_ref.rollout.mode="async" \
actor_rollout_ref.rollout.disable_log_stats=False \
actor_rollout_ref.rollout.prometheus.enable=True

# 3. 启用性能分析
actor_rollout_ref.actor.profile.use_profile=True \
actor_rollout_ref.actor.profile.save_path="./profile"

# 4. 控制验证日志量 (避免数据爆炸)
trainer.log_val_generations=10

# 总结

| 层级        | 工具                   | 收集内容           | 用途         |
| --------- | -------------------- | -------------- | ---------- |
| **实验跟踪**  | wandb/mlflow/swanlab | 算法指标 + 系统吞吐    | 效果评估、超参对比  |
| **系统监控**  | Prometheus + Grafana | vLLM 实时状态、请求分布 | **性能瓶颈定位** |
| **性能分析**  | torch.profiler       | 计算/通信时间分解      | 细粒度优化      |
| **分布式监控** | Ray Dashboard        | 集群资源、任务调度      | 集群健康度      |

verl 的指标收集设计体现了 "算法效果"与"系统效率"并重 的理念——不仅关注模型是否学得好（reward/accuracy），更关注训练是否跑得高效（throughput/MFU），这是大规模 RL 训练工程化的关键。